In [1]:
import sys
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import itertools
import os
import random
import multiprocessing
import sys
import subprocess

BIO_LIP_COLUMNS = [
"PDB ID",
"Receptor chain",
"Resolution. '-1.00' stands for lack of resolution information, e.g. for NMR",
"Binding site number code",
"Ligand_ID",
"Ligand_chain",
"Ligand serial number",
"    Binding site residues (with PDB residue numbering)",
"    Binding site residues (with residue re-numbered starting from 1)",
"Catalytic site residues (different sites are separated by ';') (with PDB residue numbering)",
"    Catalytic site residues (different sites are separated by ';') (with residue re-numbered starting from 1)",
"EC number",
"GO terms",
"Binding affinity by manual survey of the original literature. The information in '()' is the PubMed ID",
"Binding affinity provided by the Binding MOAD database. The information in '()' is the ligand information in Binding MOAD",
"Binding affinity provided by the PDBbind-CN database. The information in '()' is the ligand information in PDBbind-CN",
"Binding affinity provided by the BindingDB database",
"UniProt ID",
"PubMed ID",
"Residue sequence number of the ligand (field _atom_site.auth_seq_id in PDBx/mmCIF format)",
"Receptor sequence"]



# now the invalid ligands are stored as strings, written one by one.
INVALID_LIGANDS = ['144', '15P', '1PE', '2F2', '2JC', '3HR', '3SY', '7N5', '7PE', '9JE', 'AAE', 'ABA', 'ACE', 'ACN', 'ACT', 'ACY', 'AZI', 'BAM', 'BCN', 'BCT', 'BDN', 'BEN', 'BME', 'BO3', 'BTB', 'BTC', 'BU1', 'C8E', 'CAD', 'CAQ', 'CBM', 'CCN', 'CIT', 'CL',
'CM', 'CMO', 'CO3', 'CPT', 'CXS', 'D10', 'DEP', 'DIO', 'DMS', 'DN', 'DOD', 'DOX', 'EDO', 'EEE', 'EGL', 'EOH', 'EOX', 'EPE', 'ETF', 'FCY', 'FJO', 'FLC', 'FMT', 'FW5', 'GOL', 'GSH', 'GTT', 'GYF', 'HED', 'IHP', 'IHS', 'IMD', 'IOD', 'IPA', 'IPH',
'LDA', 'MB3', 'MEG', 'MES', 'MLA', 'MLI', 'MOH', 'MPD', 'MRD', 'MSE', 'MYR', 'N', 'NA', 'NH2', 'NH4', 'NHE', 'NO3', 'O4B', 'OHE', 'OLA', 'OLC', 'OMB', 'OME', 'OXA', 'P6G', 'PE3', 'PE4', 'PEG', 'PEO', 'PEP', 'PG0', 'PG4', 'PGE', 'PGR',
'PLM', 'PO4', 'POL', 'POP', 'PVO', 'SAR', 'SCN', 'SEO', 'SEP', 'SIN', 'SO4', 'SPD', 'SPM', 'SR', 'STE', 'STO', 'STU', 'TAR', 'TBU', 'TME', 'TPO', 'TRS', 'UNK', 'UNL', 'UNX', 'UPL', 'URE']


# now with these: SO4, GOL, EDO, PO4, ACT, PEG, DMS, TRS, PGE, PG4, FMT, EPE, MPD, MES, CD, IOD
CRYSTALIZATION_LIGANDS = ['SO4', 'GOL', 'EDO', 'PO4', 'ACT', 'PEG', 'DMS', 'TRS', 'PGE', 'PG4', 'FMT', 'EPE', 'MPD', 'MES', 'CD', 'IOD']

ALL_INVALID_LIGANDS = INVALID_LIGANDS + CRYSTALIZATION_LIGANDS

# path2biolip = '/home/iscb/wolfson/hagairavid/LocAlign/BioLiP_nr.txt'
path2biolip = '/Users/jerometubiana/Downloads/BioLiP.txt'
path2biolip_ligand = '/Users/jerometubiana/Downloads/ligand.tsv'

df = pd.read_csv(path2biolip, sep="\t", header=None, names=BIO_LIP_COLUMNS)
relevant_columns = ["PDB ID", "Receptor chain", "Ligand_ID", "Ligand_chain",'Receptor sequence',
                    "Resolution. '-1.00' stands for lack of resolution information, e.g. for NMR",
                    "    Binding site residues (with PDB residue numbering)"]
df = df[relevant_columns].astype('str')
print(df.shape)
df = df[~df['Ligand_ID'].str.contains('DNA|RNA|PEPTIDE|NONE|nan', case=False, na=False)]
print(df.shape)
df = df[~df['Ligand_ID'].map(lambda x: x in ALL_INVALID_LIGANDS)]
print(df.shape)
df = df[ (df["Resolution. '-1.00' stands for lack of resolution information, e.g. for NMR"].astype(float) < 4.)]
print(df.shape)
df = df[~df['Receptor chain'].map(lambda x: len(x)>1)]
print(df.shape)

df = df[ df['Receptor sequence'].map(lambda x: len(x) <=1024) ]
print(df.shape)

# df = df[:1000:10]
df = df.reset_index(drop=True)


/var/folders/52/fmf_vkm95_s5lgbwbnzb1yb80000gn/T/ipykernel_82329/652169267.py:53: DtypeWarning: Columns (9,10,13,14,16) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path2biolip, sep="\t", header=None, names=BIO_LIP_COLUMNS)


(989058, 7)
(713554, 7)
(697139, 7)
(664746, 7)
(604097, 7)
(590249, 7)


In [2]:
## Add ligand metadata

def atom_counts(formula):
    formula_split = formula.strip().split(' ')
    count_atoms = 0
    for f in formula_split:
        if f.startswith('H'):
            continue
        else:
            numerics = ''.join(alpha for alpha in f if alpha.isdigit())
            if len(numerics)>0:
                count_atoms += int(numerics)
            else:
                count_atoms+=1
    return count_atoms
                                
ligand_df = pd.read_csv(path2biolip_ligand,sep='\t',index_col='#CCD',usecols=['#CCD','formula'])
ligand_df['n_atoms'] = ligand_df['formula'].map(atom_counts)


df['src_motif'] = df['    Binding site residues (with PDB residue numbering)'].map(
    lambda annotation:  '[' +  ','.join([ x[1:] for x in  annotation.split(' ') ]) + ']')

df['src_ligand_n_residues'] = 1
df['src_ligand_n_atoms'] = df['Ligand_ID'].map(lambda x: ligand_df['n_atoms'].get(x,None))
df['motif_size'] = df['    Binding site residues (with PDB residue numbering)'].map(lambda annotation:  len(annotation.split(' ')) )





In [3]:

import shutil

def cluster_sequences(list_sequences, seqid=1.0, coverage=0.8, covmode='0', path2mmseqstmp=None,
                      path2mmseqs=None, threads=8
                      ):

    # Use a Linux-friendly temp directory (defaults to $TMPDIR or a local folder)
    if path2mmseqstmp is None:
        path2mmseqstmp = os.environ.get('TMPDIR', os.path.join(os.getcwd(), 'tmp_mmseqs'))
    os.makedirs(path2mmseqstmp, exist_ok=True)

    # Resolve mmseqs executable
    if path2mmseqs is None:
        mmseqs_exec = shutil.which('mmseqs')
        if mmseqs_exec is None:
            raise FileNotFoundError(
                "mmseqs executable not found in PATH. Install mmseqs2 (e.g., 'conda install -c bioconda mmseqs2') or pass path2mmseqs explicitly."
            )
        path2mmseqs = mmseqs_exec
    else:
        if not os.path.exists(path2mmseqs):
            raise FileNotFoundError(f"mmseqs executable not found at provided path: {path2mmseqs}")

    rng = np.random.randint(0, high=int(1e6))
    tmp_input = os.path.join(path2mmseqstmp, f'tmp_input_file_{rng}.fasta')
    tmp_output = os.path.join(path2mmseqstmp, f'tmp_output_file_{rng}')

    with open(tmp_input, 'w') as f:
        for k, sequence in enumerate(list_sequences):
            f.write(f'>{k}\n')
            f.write(f'{sequence}\n')

    command = ('{mmseqs} easy-cluster {fasta} {result} {tmp} --threads {threads} --min-seq-id %s -c %s --cov-mode %s' % (
        seqid, coverage, covmode)).format(threads=threads, mmseqs=path2mmseqs, fasta=tmp_input, result=tmp_output, tmp=path2mmseqstmp)
    subprocess.run(command.split(' '), check=True)

    with open(tmp_output + '_rep_seq.fasta', 'r') as f:
        representative_indices = [int(x[1:-1]) for x in f.readlines()[::2]]
    cluster_indices = np.zeros(len(list_sequences), dtype=int)
    table_path = tmp_output + '_cluster.tsv'
    table = pd.read_csv(table_path, sep='\t', header=None).to_numpy(dtype=int)
    for i, j in table:
        if i in representative_indices:
            cluster_indices[j] = representative_indices.index(i)
    # Cleanup temporary files if they exist
    for file in [tmp_output + '_rep_seq.fasta', tmp_output + '_all_seqs.fasta', tmp_output + '_cluster.tsv', tmp_input]:
        try:
            if os.path.exists(file):
                os.remove(file)
        except Exception:
            pass
    return np.array(cluster_indices), np.array(representative_indices)


cluster_indices, representative_indices = cluster_sequences(df['Receptor sequence'].to_list(), seqid=0.9,coverage=0.9)

    
resolution = df["Resolution. '-1.00' stands for lack of resolution information, e.g. for NMR"].astype(float)
resolution[resolution<0] = 4. # Assign 4A res to all NMR ensembles.

importance = df['motif_size'].astype(float) + df['src_ligand_n_atoms'].astype(float) - resolution 
# For each cluster, take as representative the structure with best resolution and/or largest motif

representative_indices = [] 
for cluster in np.unique(cluster_indices):
    res_ = (cluster_indices== cluster) * resolution + (cluster_indices != cluster) * 10000.
    representative_indices.append( res_.argmin() )
representative_indices = np.array(representative_indices)
    
df['cluster_indices'] = cluster_indices

df['id'] = df['PDB ID'] + '_' + df['Receptor chain']

all_ligands = df.groupby('cluster_indices')['Ligand_ID'].unique().map(lambda x: '|'.join(x))
all_ids = df.groupby('cluster_indices')['id'].unique().map(lambda x: '|'.join(x))

df['Ligand_all']=all_ligands[df['cluster_indices']].reset_index(drop=True)
df['id_all']=all_ids[df['cluster_indices']].reset_index(drop=True)
df_nr = df.iloc[representative_indices].reset_index(drop=True)
print(df.shape, df_nr.shape)

easy-cluster /var/folders/52/fmf_vkm95_s5lgbwbnzb1yb80000gn/T/tmp_input_file_549667.fasta /var/folders/52/fmf_vkm95_s5lgbwbnzb1yb80000gn/T/tmp_output_file_549667 /var/folders/52/fmf_vkm95_s5lgbwbnzb1yb80000gn/T/ --threads 8 --min-seq-id 0.9 -c 0.9 --cov-mode 0 

MMseqs Version:                     	14-7e284
Substitution matrix                 	aa:blosum62.out,nucl:nucleotide.out
Seed substitution matrix            	aa:VTML80.out,nucl:nucleotide.out
Sensitivity                         	4
k-mer length                        	0
k-score                             	seq:2147483647,prof:2147483647
Alphabet size                       	aa:21,nucl:5
Max sequence length                 	65535
Max results per query               	20
Split database                      	0
Split mode                          	2
Split memory limit                  	0
Coverage threshold                  	0.9
Coverage mode                       	0
Compositional bias                  	1
Compositional bias              

In [4]:
df_nr

,PDB ID,Receptor chain,Ligand_ID,Ligand_chain,Receptor sequence,"Resolution. '-1.00' stands for lack of resolution information, e.g. for NMR",Binding site residues (with PDB residue numbering),src_motif,src_ligand_n_residues,src_ligand_n_atoms,motif_size,cluster_indices,id,Ligand_all,id_all
0,12as,A,ASN,A,AYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLS...,2.2,D46 S72 A74 K77 D118 Y218 R255 G294,"[46,72,74,77,118,218,255,294]",1,9.0,8,0,12as_A,ASN|AMP,11as_A|11as_B|12as_A|12as_B
1,5j41,A,3LF,A,PPYTVVYFPVRGRCAALRMLLADQGQSWKEEVVTVETWQEGSLKAS...,1.19035,F8 R13 I104 Y108,"[8,13,104,108]",1,17.0,4,1,5j41_A,VWW|EAA|0HH|SAS|GTX|GDN|BSP|0HG|GTD|LEE|NO|CBD...,10gs_A|10gs_B|11gs_A|11gs_B|12gs_A|12gs_B|13gs...
2,16pk,A,BIS,A,EKKSINECDLKGKKVLIRVDFNVPVKNGKITNDYRIRSALPTLKKV...,1.6,G217 A218 K223 A242 Y245 L315 P340 G342 V343 E...,"[217,218,223,242,245,315,340,342,343,345,397,3...",1,39.0,13,2,16pk_A,ADP|BIS,13pk_A|13pk_B|13pk_C|13pk_D|16pk_A
3,155c,A,HEM,A,NEGDAAKGEKEFNKCKACHMIQAPDGTDIKGGKTGPNLYGVVGRKI...,2.5,C15 C18 H19 I46 A47 Y54 G55 I58 W70 Y78 V79 K9...,"[15,18,19,46,47,54,55,58,70,78,79,98,99]",1,43.0,13,3,155c_A,HEM,155c_A
4,1ofw,A,HEC,A,AALEPTDSGAPSAIVMFPVGEKPNPKGAAMKPVVFNHLIHEKKIDN...,1.5,S8 A10 I14 F35 H37 H40 I44 C47 C50 H51 P56 V57...,"[8,10,14,35,37,40,44,47,50,51,56,57,58]",1,43.0,13,4,1ofw_A,HEM|HEC,19hc_A|19hc_B|1ofw_A|1ofw_B|1ofy_A|1ofy_B
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32645,4h15,A,CA,A,SMMIEFLNLRGKRALITAGTKGAGAATVSLFLELGAQVLTTARARP...,1.45,A96 D101,"[96,101]",1,1.0,2,32645,4h15_A,CA|MG,4h15_A|4h15_B|4h15_C|4h15_D|4h16_A
32646,4h19,A,CA,A,MKITAVEPFILHLPLTSESISDSTHSITHWGVVGAKITTSDGIEGY...,1.8,D268 H298,"[268,298]",1,1.0,2,32646,4h19_A,CA|MG|0YR,4h19_A|4h19_B|4h19_C|4h19_D|4h19_E|4h19_F|4h19...
32647,4h1g,A,GLC,B,KIEEGKLVIWINGDKGYNGLAEVGKKFEKDTGIKVTVEHPDKLEEK...,2.15,D-356 K-355 E-259 Y-215 W-140,"[-356,-355,-259,-215,-140]",1,12.0,5,32647,4h1g_A,GLC|ADP,4h1g_A
32648,3w6p,A,GDP,A,GPMEALIPVINKLQDVFNTVGADIIQLPQIVVVGTQSSGKSSVLES...,1.7,S35 G37 K38 S39 S40 R53 G54 T55 K216 D218 L219...,"[35,37,38,39,40,53,54,55,216,218,219,245,246,2...",1,28.0,16,32648,3w6p_A,GNH|MG|GDP|ALF|GNP|6I9,3w6n_A|3w6n_B|3w6o_A|3w6o_B|3w6p_A|3w6p_B|4h1v...


In [11]:
df_nr = df.iloc[representative_indices].reset_index(drop=True)
df_nr = df_nr.drop(columns=['Receptor sequence','Ligand_chain','id',
                            "    Binding site residues (with PDB residue numbering)"]).rename(columns={'PDB ID':'src_protein',
                                                                         'Receptor chain':'src_chain',
                                                                         'Ligand_ID':'src_ligand',
                                                                         'Ligand_all':'ligand_all'})
# df_nr['src_motif'] = [None for _ in range(len(df_nr))]
df_nr = df_nr[ ['src_ligand','src_chain','src_protein','src_motif','src_ligand_n_atoms','src_ligand_n_residues',
                'ligand_all','id_all'] ]
df_nr['src_ligand_n_atoms'] = df_nr['src_ligand_n_atoms'].map(lambda x: '[' + (str(int(x)) if ~np.isnan(x) else '') + ']')
df_nr_no_motif = df_nr.copy()
df_nr_no_motif['src_motif'] = [None for _ in range(len(df_nr_no_motif))]

df_nr_no_motif.to_csv('../example_inputs/biolip2_nr_database.csv',index=False)
df_nr_no_motif.iloc[:100].to_csv('../example_inputs/biolip2_nr_mini_database.csv')

df_nr.to_csv('../example_inputs/biolip2_nr_database_with_motif.csv',index=False)
df_nr.iloc[:100].to_csv('../example_inputs/biolip2_nr_mini_database_with_motif.csv')

In [10]:
df_nr

,src_ligand,src_chain,src_protein,src_motif,src_ligand_n_atoms,src_ligand_n_residues,ligand_all,id_all
0,ASN,A,12as,"[46,72,74,77,118,218,255,294]",[9],1,ASN|AMP,11as_A|11as_B|12as_A|12as_B
1,3LF,A,5j41,"[8,13,104,108]",[17],1,VWW|EAA|0HH|SAS|GTX|GDN|BSP|0HG|GTD|LEE|NO|CBD...,10gs_A|10gs_B|11gs_A|11gs_B|12gs_A|12gs_B|13gs...
2,BIS,A,16pk,"[217,218,223,242,245,315,340,342,343,345,397,3...",[39],1,ADP|BIS,13pk_A|13pk_B|13pk_C|13pk_D|16pk_A
3,HEM,A,155c,"[15,18,19,46,47,54,55,58,70,78,79,98,99]",[43],1,HEM,155c_A
4,HEC,A,1ofw,"[8,10,14,35,37,40,44,47,50,51,56,57,58]",[43],1,HEM|HEC,19hc_A|19hc_B|1ofw_A|1ofw_B|1ofy_A|1ofy_B
...,...,...,...,...,...,...,...,...
32645,CA,A,4h15,"[96,101]",[1],1,CA|MG,4h15_A|4h15_B|4h15_C|4h15_D|4h16_A
32646,CA,A,4h19,"[268,298]",[1],1,CA|MG|0YR,4h19_A|4h19_B|4h19_C|4h19_D|4h19_E|4h19_F|4h19...
32647,GLC,A,4h1g,"[-356,-355,-259,-215,-140]",[12],1,GLC|ADP,4h1g_A
32648,GDP,A,3w6p,"[35,37,38,39,40,53,54,55,216,218,219,245,246,2...",[28],1,GNH|MG|GDP|ALF|GNP|6I9,3w6n_A|3w6n_B|3w6o_A|3w6o_B|3w6p_A|3w6p_B|4h1v...


In [7]:
import sys
sys.path.append('../')

from miners.objects import Protein
from multiprocessing import Pool, cpu_count


def get_pocket_residue_ids(protein_name: str, chain: str, ligand: str, ligand_res_idx: int = 0, distance_thresh: float = 4.0) -> list[int]:
    """
    Extract pocket residue IDs for a given protein-chain-ligand combination.
    Aggregates across all ligand residues (if multiple) and returns a single
    flat, sorted list of unique residue IDs.
    
    Args:
        protein_name: PDB protein name (e.g., '1gkm')
        chain: Chain identifier (e.g., 'A')
        ligand: Ligand identifier (e.g., 'general', 'ADP', etc.)
        ligand_res_idx: Ignored; kept for backward compatibility
        distance_thresh: Pocket cutoff in Å (default: 4.0)
    
    Returns:
        List of residue IDs in the pocket (unique, sorted)
    """
    try:
        protein = Protein(protein_name, chain, ligand, save_models=False)
        ligand_residues = protein.get_ligand_residues()
        pockets = set()
        for i in range(len(ligand_residues)):
            ids = protein.get_pocket_residue_ids(distance_thresh=distance_thresh, ligand_res_idx=i)
            for rid in ids:
                try:
                    pockets.add(int(rid))
                except Exception:
                    pockets.add(rid)
        return sorted(pockets) if pockets else None
    except Exception as e:
        print(f"Error getting pocket for {protein_name} chain {chain} ligand {ligand}: {e}")
        return None


def compute_pocket_for_tuple(args_tuple: tuple[str, str, str], distance_thresh: float = 4.0) -> tuple[list[int] | None, int]:
    """
    Multiprocessing-safe wrapper to compute pocket residue IDs aggregating over
    all ligand residues for a given protein/chain/ligand.
    Returns (sorted_pocket_ids_or_None, ligand_residue_count, n_ligand_atoms_list).
    """
    protein_name, chain, ligand = args_tuple
    try:
        protein = Protein(protein_name, chain, ligand, save_models=False)
        ligand_residues = protein.get_ligand_residues()
        residue_count = len(ligand_residues)
        n_ligand_atoms = [len(list(residue.get_atoms())) for residue in ligand_residues]
        # print(f"Processing {protein_name} {chain} {ligand} with {residue_count} ligand residues and n_ligand_atoms: {n_ligand_atoms}")
        pockets = set()
        for i in range(residue_count):
            ids = protein.get_pocket_residue_ids(distance_thresh=distance_thresh, ligand_res_idx=i)
            for rid in ids:
                try:
                    pockets.add(int(rid))
                except Exception:
                    pockets.add(rid)
        return (sorted(pockets) if pockets else None), residue_count, n_ligand_atoms
    except Exception as e:
        print(f"Error (MP) processing {protein_name} {chain} {ligand}: {e}")
        return None, 0, []

# Quick test
test_pockets, test_count, test_n_ligand_atoms = compute_pocket_for_tuple(('1gkm', 'A', 'general'))
print(f"Test pockets: {test_pockets} | ligand residues: {test_count} | n_ligand_atoms: {test_n_ligand_atoms}")

Error (MP) processing 1gkm A general: 'A'
Test pockets: None | ligand residues: 0 | n_ligand_atoms: []


In [11]:
# Apply pocket residue extraction to all proteins in df_nr (multiprocessing)
print(f"Processing {len(df_nr)} proteins to extract pocket residues (multiprocessing)...")

args_list = [(row['src_protein'], row['src_chain'], row['src_ligand']) for _, row in df_nr.iterrows()]
pocket_residues_list = []
ligand_residue_counts = []
n_ligand_atoms_list = []

if args_list:
    processes = 1#min(len(args_list), max(1, cpu_count() - 1))
    with Pool(processes=processes) as pool:
        # Preserve order using imap
        # for i, (pockets, count, n_ligand_atoms) in enumerate(pool.imap(compute_pocket_for_tuple, args_list), 1):
        for i, (pockets, count, n_ligand_atoms) in enumerate(map(compute_pocket_for_tuple, args_list), 1):
            pocket_residues_list.append(pockets)
            ligand_residue_counts.append(count)
            n_ligand_atoms_list.append(n_ligand_atoms)
            if i % 50 == 0:
                print(f"Progress: {i}/{len(args_list)}")

# Set columns: src_motif equals pocket_residue_ids, add ligand residue count
df_nr['pocket_residue_ids'] = pocket_residues_list
df_nr['src_motif'] = df_nr['pocket_residue_ids']
df_nr['src_ligand_n_residues'] = ligand_residue_counts
df_nr['src_ligand_n_atoms'] = n_ligand_atoms_list

multi_res_cases = (df_nr['src_ligand_n_residues'] > 1).sum()
print(f"\nCompleted! Added pocket_residue_ids, src_motif, num_ligand_residues. Multi-residue cases: {multi_res_cases}")
print(f"Sample row: {df_nr.iloc[0]}")

Processing 32650 proteins to extract pocket residues (multiprocessing)...


Failed to download 12as.cif from https://files.rcsb.org/download/12as.cif
Failed to download 5j41.cif from https://files.rcsb.org/download/5j41.cif
Failed to download 16pk.cif from https://files.rcsb.org/download/16pk.cif
Failed to download 155c.cif from https://files.rcsb.org/download/155c.cif


Error (MP) processing 12as A ASN: 0
Error (MP) processing 5j41 A 3LF: 0
Error (MP) processing 16pk A BIS: 0
Error (MP) processing 155c A HEM: 0


Failed to download 1ofw.cif from https://files.rcsb.org/download/1ofw.cif
Failed to download 1a05.cif from https://files.rcsb.org/download/1a05.cif
Failed to download 2a0b.cif from https://files.rcsb.org/download/2a0b.cif


Error (MP) processing 1ofw A HEC: 0
Error (MP) processing 1a05 A IPM: 0
Error (MP) processing 2a0b A ZN: 0


...

Failed to download 1afj.cif from https://files.rcsb.org/download/1afj.cif
Failed to download 1gg6.cif from https://files.rcsb.org/download/1gg6.cif
Failed to download 1ag6.cif from https://files.rcsb.org/download/1ag6.cif


Error (MP) processing 1afj A HG: 0
Error (MP) processing 1gg6 B APL: 0
Error (MP) processing 1ag6 A CU: 0


Failed to download 1d1t.cif from https://files.rcsb.org/download/1d1t.cif


Error (MP) processing 1d1t A ZN: 0


KeyboardInterrupt: 

In [ ]:
# Save the final dataframe with pocket residue IDs

# Ensure src_motif column entries are lists (or None)
df_nr['src_motif'] = df_nr['src_motif'].apply(lambda x: x if (x is None or isinstance(x, list)) else [x])

df_nr = df_nr.drop(columns=['pocket_residue_ids'])

# save full
df_nr.to_csv('../example_inputs/biolip2_nr_database_with_motif.csv', index=False)
print("✅ Saved df_nr with pocket residue IDs and src_motif lists to 'biolip2_nr_database_with_motif.csv'")

# save mini
df_nr.head(50).to_csv('../example_inputs/biolip2_nr_mini_database_with_motif.csv', index=False)
print("✅ Saved df_nr mini to 'biolip2_nr_mini_database_with_motif.csv'")

print(f"\nDataframe shape: {df_nr.shape}")
print(f"Columns: {list(df_nr.columns)}")
print("src_motif types:", df_nr['src_motif'].apply(lambda x: type(x).__name__).value_counts().to_dict())
print(f"\nFirst few rows:")
df_nr.head()

✅ Saved df_nr with pocket residue IDs and tar_motif lists to 'biolip2_nr_database_with_motif_n_ligand_atoms.csv'
✅ Saved df_nr mini to 'biolip2_nr_mini_database_with_motif_n_ligand_atoms.csv'

Dataframe shape: (23035, 8)
Columns: ['tar_ligand', 'tar_chain', 'tar_protein', 'tar_motif', 'ligand_all', 'id_all', 'num_ligand_residues', 'n_ligand_atoms']
tar_motif types: {'list': 21151, 'NoneType': 1884}

First few rows:


,tar_ligand,tar_chain,tar_protein,tar_motif,ligand_all,id_all,num_ligand_residues,n_ligand_atoms
0,ASN,A,11as,"[45, 46, 48, 49, 50, 52, 72, 74, 75, 77, 100, ...",ASN,11as_A,4,"[8, 8, 8, 9]"
1,CS,A,1kh8,"[58, 73, 113, 114, 115]",UPA|CS,11ba_A|1kh8_A,1,[1]
2,ADP,A,13pk,"[217, 218, 219, 223, 241, 242, 245, 294, 314, ...",ADP,13pk_A,1,[27]
3,HEM,A,19hc,"[8, 10, 11, 14, 16, 17, 18, 29, 30, 31, 32, 33...",HEM|HEC,19hc_A|1duw_A,9,"[43, 43, 43, 43, 43, 43, 43, 43, 43]"
4,IPM,A,1a05,"[88, 95, 105, 133, 140, 246, 501, 524, 553, 56...",IPM,1a05_A,1,[12]


In [ ]:
'''
Manual fixes:
1gkm_A => 1gkm_B                                                                                                 | 1/8 [00:11<01:19, 11.33s/it]Failed to download 4ct3 chain A ligand general: 'A'
4ct3_A => 4ct3_E
2h8p_D => delete
'''

In [ ]:
# create mini